In [2]:
import logging
import os
import random
from pathlib import Path

import hydra
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyvips
import torch
from lightning import seed_everything
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from explainability.cams import grad_cam, grad_cam_pp, layer_cam
from explainability.cams.abstract import (
    HookedModule,
    modify_ReLU_inplace,
    reshape_for_clustering,
)
from explainability.precomputing import ClusteringManager
from prostate_cancer.data import DataModule
from prostate_cancer.prostate_cancer_model import ProstateCancerModel

In [3]:
def get_clustering_instance_fp(clustering_algorithm: str, num_clusters: int) -> Path:
    return Path(
        "/mnt/projects/explainability/XAICNNEmbeddings/PRECOMPUTED",
        "VGG16_Prostate",
        "global_clustering_instances",
        f"clustering_instance_test_global_{clustering_algorithm.lower()}_{num_clusters}clusters.npy",
    )

In [4]:
# Define clustering algorithms and cluster numbers to test
CLUSTERING_ALGORITHMS = ["NMF", "KMeans"]
CLUSTER_NUMBERS = [4, 6, 8]

In [5]:
# Load all clustering models
print("Loading all clustering models...")
clustering_models = {}

for clustering_algorithm in CLUSTERING_ALGORITHMS:
    for num_clusters in CLUSTER_NUMBERS:
        model_name = f"{clustering_algorithm}_{num_clusters}"
        print(f"Loading {model_name}...")

        clustering_instance_fp = get_clustering_instance_fp(
            clustering_algorithm, num_clusters
        )
        clustering_model = ClusteringManager.load_model(
            algorithm=clustering_algorithm,
            num_clusters=num_clusters,
            path=clustering_instance_fp,
        )

        clustering_models[model_name] = (
            clustering_model,
            num_clusters,
            clustering_algorithm,
        )

print(f"\nLoaded {len(clustering_models)} clustering models")
print(f"Models: {list(clustering_models.keys())}\n")

Loading all clustering models...
Loading NMF_4...
Loading NMF_6...
Loading NMF_8...
Loading KMeans_4...
Loading KMeans_6...
Loading KMeans_8...

Loaded 6 clustering models
Models: ['NMF_4', 'NMF_6', 'NMF_8', 'KMeans_4', 'KMeans_6', 'KMeans_8']



In [6]:
logging.basicConfig(level=logging.INFO)

# Set random seed for reproducibility
seed_everything(42, workers=True)
torch.set_float32_matmul_precision(precision="medium")

# Configuration overrides for prediction
overrides = ["experiment=predict/images/vgg16", "mode=predict"]

# Initialize Hydra configuration
with hydra.initialize(config_path="conf", version_base=None):
    config = hydra.compose(config_name="default", overrides=overrides)

print("Configuration loaded successfully!")
print(f"Mode: {config.mode}")
print(f"Checkpoint: {config.checkpoint}")
print(f"Batch size: {config.data.batch_size}")

# Instantiate data module and model
data: DataModule = hydra.utils.instantiate(
    config.data,
    _recursive_=False,  # to avoid instantiating all the datasets
    _target_=DataModule,
)

chkcpt_path = mlflow.artifacts.download_artifacts(config.checkpoint)
checkpoint = torch.load(chkcpt_path, map_location="cuda:0")

model: ProstateCancerModel = hydra.utils.instantiate(config.model)
model.load_state_dict(checkpoint["state_dict"], strict=True)

model = model.to("cuda:0")

print("Data module and model instantiated successfully!")

print(model)

Seed set to 42


Configuration loaded successfully!
Mode: predict
Checkpoint: mlflow-artifacts:/65/7b52930515c14710855962f8882fb4d3/artifacts/checkpoints/epoch=1-step=11620/checkpoint.ckpt
Batch size: 74


Data module and model instantiated successfully!
ProstateCancerModel(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
 

In [7]:
mlflow_client = MlflowClient(tracking_uri="http://mlflow.rationai-mlflow:5000/")
chkcpt_path = mlflow_client.download_artifacts(
    run_id="199141b76f90457fbf6183bd70a1e488", path="annotation_masks"
)

In [8]:
def get_carcinoma_mask(slide_id: str) -> np.ndarray | None:
    if not os.path.exists(f"{chkcpt_path}/carcinoma/{slide_id}.tiff"):
        return None

    carcinoma = pyvips.Image.new_from_file(f"{chkcpt_path}/carcinoma/{slide_id}.tiff")
    carcinoma_np = np.asarray(carcinoma) > 0

    # if exclude does not exists, it is empty
    if not os.path.exists(f"{chkcpt_path}/exclude/{slide_id}.tiff"):
        exclude_np = np.zeros_like(carcinoma_np)
    else:
        exclude = pyvips.Image.new_from_file(f"{chkcpt_path}/exclude/{slide_id}.tiff")
        exclude_np = np.asarray(exclude) > 0

    return carcinoma_np & ~exclude_np


def get_another_pathology_mask(slide_id: str) -> np.ndarray | None:
    if not os.path.exists(f"{chkcpt_path}/another_pathology/{slide_id}.tiff"):
        return None

    another_pathology = pyvips.Image.new_from_file(
        f"{chkcpt_path}/another_pathology/{slide_id}.tiff"
    )
    return np.asarray(another_pathology) > 0


def get_tile_mask(mask: np.ndarray | None, x: int, y: int) -> np.ndarray | None:
    if mask is None:
        return None

    yy, xx = x // 16, y // 16
    # get the 32x32 patch around th
    tile_mask = mask[xx : xx + 32, yy : yy + 32]

    if tile_mask.sum() == 0:
        return None

    return tile_mask

In [9]:
target_layer = "backbone.29"
hooked_model = HookedModule(model, layer_names=[target_layer])
modify_ReLU_inplace(hooked_model, inplace=False)

In [10]:
# Get one batch from validation dataset
data.batch_size = 32
data.setup("test")
dataloaders = data.test_dataloader()

In [11]:
carcinoma_masks = {}
another_pathology_masks = {}

for i in range(len(dataloaders)):
    slide_metadata = data.test.slides.iloc[i]
    extent_x = slide_metadata["extent_x"]
    extent_y = slide_metadata["extent_y"]
    slide_id = Path(slide_metadata["path"]).stem

    carcinoma_masks[slide_id] = get_carcinoma_mask(slide_id)
    another_pathology_masks[slide_id] = get_another_pathology_mask(slide_id)

INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 7 workers
INFO:pyvips:VIPS: threadpool completed with 7 workers
INFO:pyvips:VIPS: threadpool completed with 6 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 6 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 6 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 4 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool

In [12]:
# Constants
CAM_METHODS = ["grad_cam", "grad_cam_pp", "layer_cam"]
MASK_TYPES = ["carcinoma", "another_pathology"]


def create_data(clustering_models, tile_samples):
    all_datasets = {model_name: [] for model_name in clustering_models}

    # Process all tile samples
    for slide_index, tile_idx in tile_samples:
        dataloader = dataloaders[slide_index]
        dataset = dataloader.dataset
        tile, label, metadata = dataset[tile_idx]
        tile = tile.to("cuda:0").unsqueeze(0)

        # Forward pass - compute once for all models
        output = hooked_model(tile)
        output.backward()

        # Get activations and gradients once
        activations = hooked_model.get_activations(target_layer)
        activations_flattened, shape = reshape_for_clustering(activations)
        activations_np = activations_flattened.detach().cpu().numpy()
        gradients = hooked_model.get_gradients(target_layer)

        slide, x, y = metadata["slide"], metadata["x"], metadata["y"]
        carcinoma_mask = get_tile_mask(carcinoma_masks[slide], x, y)
        another_pathology_mask = get_tile_mask(another_pathology_masks[slide], x, y)

        cams = {}
        cam_functions = {
            "grad_cam_pp": grad_cam_pp,
            "grad_cam": grad_cam,
            "layer_cam": layer_cam,
        }
        for cam_name in CAM_METHODS:
            cam_fn = cam_functions[cam_name]
            cam = cam_fn(activations, gradients)[0]
            cam_np = cam.flatten().detach().cpu().numpy()
            cam_mask = (cam > 1e-9).detach().cpu().numpy()
            cams[cam_name] = {
                "cam": cam,
                "cam_np": cam_np,
                "cam_mask": cam_mask,
                "cam_count": cam_mask.sum(),
            }

        batch_size, _, height, width = shape

        # Process each clustering model
        for model_name, (
            clustering_model,
            num_clusters,
            clustering_algorithm,
        ) in clustering_models.items():
            # Build header for this specific model (only columns for its num_clusters)
            header = ["label", "output", "clustering_algorithm", "num_clusters"]

            # Add columns for this model's clusters
            for i in range(num_clusters):
                header.append(f"weight_sum_{i}")
                header.append(f"cluster_count_{i}")
                header.append(f"cluster_soft_count_{i}")

            # Annotation mask presence flags
            header += ["has_carcinoma_mask", "has_another_pathology_mask"]

            # CAM columns
            for cam in CAM_METHODS:
                header += [f"{cam}_count"]
                for i in range(num_clusters):
                    header += [
                        f"intersection_count_{cam}_{i}",
                        f"union_count_{cam}_{i}",
                    ]
                    header += [
                        f"intersection_soft_count_{cam}_{i}",
                        f"union_soft_count_{cam}_{i}",
                    ]
                    header += [f"correlation_{cam}_{i}"]
                    header += [f"l2_distance_{cam}_{i}"]

                # CAM with annotation masks
                header += [
                    f"intersection_count_{cam}_carcinoma",
                    f"union_count_{cam}_carcinoma",
                    f"intersection_count_{cam}_another_pathology",
                    f"union_count_{cam}_another_pathology",
                ]

            # Cluster masks with annotation masks
            for i in range(num_clusters):
                header += [
                    f"intersection_count_cluster_{i}_carcinoma",
                    f"union_count_cluster_{i}_carcinoma",
                    f"intersection_soft_count_cluster_{i}_carcinoma",
                    f"union_soft_count_cluster_{i}_carcinoma",
                    f"intersection_count_cluster_{i}_another_pathology",
                    f"union_count_cluster_{i}_another_pathology",
                    f"intersection_soft_count_cluster_{i}_another_pathology",
                    f"union_soft_count_cluster_{i}_another_pathology",
                ]

            # Transform activations with this clustering model
            if clustering_algorithm == "KMeans":
                # For KMeans, transform returns distances, use argmin to get cluster assignments
                distances = clustering_model.transform(activations_np)
                clustering_flat = distances.argmin(
                    axis=1
                )  # Cluster with minimum distance
                clustering = clustering_flat.reshape(batch_size, height, width)

                # For KMeans, we use distances as "weights" for sum calculation
                # but they represent distances, not probabilities
                weights_sum = np.sum(distances, axis=0)

                # For soft masks with KMeans, we could use inverse distances or just use hard masks
                # Using inverse distances normalized by sum for soft assignment
                epsilon = 1e-9
                inv_distances = 1.0 / (distances + epsilon)
                inv_distances_norm = inv_distances / (
                    inv_distances.sum(axis=1, keepdims=True) + epsilon
                )
                weights_reshaped = inv_distances_norm.reshape(
                    batch_size, height, width, num_clusters
                )
            else:
                # For NMF, transform returns weights/coefficients
                weights = clustering_model.transform(activations_np)
                weights_sum = np.sum(weights, axis=0)
                weights_reshaped = weights.reshape(
                    batch_size, height, width, num_clusters
                )

                clustering = weights.argmax(axis=1)
                clustering = clustering.reshape(batch_size, height, width)

            # Count number of tiles per cluster
            cluster_counts = np.bincount(clustering.flatten(), minlength=num_clusters)

            # Start building row data
            tile_data = [
                label.item(),
                output.cpu().item(),
                clustering_algorithm,
                num_clusters,
            ]

            # Add weight_sum, cluster_count, cluster_soft_count for this model's clusters
            for i in range(num_clusters):
                tile_data.append(weights_sum[i])
                tile_data.append(cluster_counts[i])
                if clustering_algorithm == "KMeans":
                    # For KMeans, count pixels where this cluster has highest inverse distance (soft assignment)
                    max_cluster = weights_reshaped[0, :, :, :].argmax(axis=2)
                    tile_data.append((max_cluster == i).sum())
                else:
                    # For NMF, count pixels where weight > threshold
                    tile_data.append((weights[:, i] > 1e-9).sum())

            # Add annotation mask presence flags
            has_carcinoma = carcinoma_mask is not None
            has_another_pathology = another_pathology_mask is not None
            tile_data.append(int(has_carcinoma))
            tile_data.append(int(has_another_pathology))

            # Create masks for this model
            masks = []
            for i in range(num_clusters):
                mask = np.zeros((height, width), dtype=np.float32)
                mask[clustering[0, :, :] == i] = 1
                assert mask.sum() == cluster_counts[i]
                assert mask.shape == (height, width)
                masks.append(mask)

            soft_masks = []
            for i in range(num_clusters):
                mask = np.zeros((height, width), dtype=np.float32)
                if clustering_algorithm == "KMeans":
                    # For KMeans, use inverse distance threshold for soft assignment
                    mask[weights_reshaped[0, :, :, i] > 1e-3] = 1
                else:
                    # For NMF, use weight threshold
                    mask[weights_reshaped[0, :, :, i] > 1e-9] = 1
                assert mask.shape == (height, width)
                soft_masks.append(mask)

            # Process CAMs
            for cam_name in CAM_METHODS:
                cam_data = cams[cam_name]
                cam_count = cam_data["cam_count"]
                cam_mask = cam_data["cam_mask"]
                cam_np = cam_data["cam_np"]

                tile_data.append(cam_count)

                for i in range(num_clusters):
                    # Use masks for this cluster
                    for cluster_mask in [masks[i], soft_masks[i]]:
                        intersection_count = (cam_mask * cluster_mask).sum()
                        union_count = (cam_mask + cluster_mask).sum()
                        tile_data.extend([int(intersection_count), int(union_count)])

                    # Correlation
                    weights_cluster = weights_reshaped[0, :, :, i].flatten()
                    if (
                        cam_count == 0
                        or np.std(cam_np) == 0
                        or np.std(weights_cluster) == 0
                    ):
                        correlation = np.nan
                    else:
                        correlation = np.corrcoef(cam_np, weights_cluster)[0, 1]
                    tile_data.append(correlation)

                    # Calculate L2 distance
                    weights_cluster_flat = weights_reshaped[0, :, :, i].flatten()
                    l2_distance = np.sqrt(np.sum((cam_np - weights_cluster_flat) ** 2))
                    tile_data.append(l2_distance)

                # CAM with annotation masks
                if has_carcinoma:
                    intersection_cam_carcinoma = (cam_mask * carcinoma_mask).sum()
                    union_cam_carcinoma = (cam_mask + carcinoma_mask).sum()
                else:
                    intersection_cam_carcinoma = 0
                    union_cam_carcinoma = 0
                tile_data.extend(
                    [int(intersection_cam_carcinoma), int(union_cam_carcinoma)]
                )

                if has_another_pathology:
                    intersection_cam_another = (cam_mask * another_pathology_mask).sum()
                    union_cam_another = (cam_mask + another_pathology_mask).sum()
                else:
                    intersection_cam_another = 0
                    union_cam_another = 0
                tile_data.extend(
                    [int(intersection_cam_another), int(union_cam_another)]
                )

            # Cluster masks with annotation masks
            for i in range(num_clusters):
                # Hard cluster mask with carcinoma
                if has_carcinoma:
                    intersection_cluster_carcinoma = (masks[i] * carcinoma_mask).sum()
                    union_cluster_carcinoma = (masks[i] + carcinoma_mask).sum()
                else:
                    intersection_cluster_carcinoma = 0
                    union_cluster_carcinoma = 0
                tile_data.extend(
                    [int(intersection_cluster_carcinoma), int(union_cluster_carcinoma)]
                )

                # Soft cluster mask with carcinoma
                if has_carcinoma:
                    intersection_soft_cluster_carcinoma = (
                        soft_masks[i] * carcinoma_mask
                    ).sum()
                    union_soft_cluster_carcinoma = (
                        soft_masks[i] + carcinoma_mask
                    ).sum()
                else:
                    intersection_soft_cluster_carcinoma = 0
                    union_soft_cluster_carcinoma = 0
                tile_data.extend(
                    [
                        int(intersection_soft_cluster_carcinoma),
                        int(union_soft_cluster_carcinoma),
                    ]
                )

                # Hard cluster mask with another_pathology
                if has_another_pathology:
                    intersection_cluster_another = (
                        masks[i] * another_pathology_mask
                    ).sum()
                    union_cluster_another = (masks[i] + another_pathology_mask).sum()
                else:
                    intersection_cluster_another = 0
                    union_cluster_another = 0
                tile_data.extend(
                    [int(intersection_cluster_another), int(union_cluster_another)]
                )

                # Soft cluster mask with another_pathology
                if has_another_pathology:
                    intersection_soft_cluster_another = (
                        soft_masks[i] * another_pathology_mask
                    ).sum()
                    union_soft_cluster_another = (
                        soft_masks[i] + another_pathology_mask
                    ).sum()
                else:
                    intersection_soft_cluster_another = 0
                    union_soft_cluster_another = 0
                tile_data.extend(
                    [
                        int(intersection_soft_cluster_another),
                        int(union_soft_cluster_another),
                    ]
                )

            assert len(tile_data) == len(header), (
                f"Expected {len(header)} columns, got {len(tile_data)}"
            )
            all_datasets[model_name].append(tile_data)

        hooked_model.zero_grad()

    # Convert each model's data list to DataFrame
    result = {}
    for model_name, data_list in all_datasets.items():
        _, num_clusters, clustering_algorithm = clustering_models[model_name]

        # Build header for this model
        header = ["label", "output", "clustering_algorithm", "num_clusters"]
        for i in range(num_clusters):
            header.append(f"weight_sum_{i}")
            header.append(f"cluster_count_{i}")
            header.append(f"cluster_soft_count_{i}")

        # Annotation mask presence flags
        header += ["has_carcinoma_mask", "has_another_pathology_mask"]

        for cam in CAM_METHODS:
            header += [f"{cam}_count"]
            for i in range(num_clusters):
                header += [f"intersection_count_{cam}_{i}", f"union_count_{cam}_{i}"]
                header += [
                    f"intersection_soft_count_{cam}_{i}",
                    f"union_soft_count_{cam}_{i}",
                ]
                header += [f"correlation_{cam}_{i}"]
                header += [f"l2_distance_{cam}_{i}"]

            # CAM with annotation masks
            header += [
                f"intersection_count_{cam}_carcinoma",
                f"union_count_{cam}_carcinoma",
                f"intersection_count_{cam}_another_pathology",
                f"union_count_{cam}_another_pathology",
            ]

        # Cluster masks with annotation masks
        for i in range(num_clusters):
            header += [
                f"intersection_count_cluster_{i}_carcinoma",
                f"union_count_cluster_{i}_carcinoma",
                f"intersection_soft_count_cluster_{i}_carcinoma",
                f"union_soft_count_cluster_{i}_carcinoma",
                f"intersection_count_cluster_{i}_another_pathology",
                f"union_count_cluster_{i}_another_pathology",
                f"intersection_soft_count_cluster_{i}_another_pathology",
                f"union_soft_count_cluster_{i}_another_pathology",
            ]

        result[model_name] = pd.DataFrame(data_list, columns=header)

    return result

In [13]:
def set_my_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

In [14]:
# set_my_seed(42)

# n = 10000

In [15]:
# # First, generate the tile samples once to ensure we use the same tiles for all modelsprint("Generating tile samples (same tiles for all models)...")
# #
# num_slides = len(dataloaders)
# slide_indices = np.random.choice(list(range(num_slides)), n, replace=True)
# slide_counts = [0] * num_slides

# for slide_index in slide_indices:
#     slide_counts[slide_index] += 1

# # Generate tile_samples: list of (slide_index, tile_idx) tuples

# tile_samples = []
# for slide_index, count in enumerate(slide_counts):
#     if count == 0:
#         continue
#     dataloader = dataloaders[slide_index]
#     dataset = dataloader.dataset
#     num_tiles_in_slide = len(dataset)
#     tile_indices = sorted(
#         np.random.choice(list(range(num_tiles_in_slide)), count, replace=True)
#     )
#     for tile_idx in tile_indices:
#         tile_samples.append((slide_index, tile_idx))


# print(f"Generated {len(tile_samples)} tile samples")
# print(
#     "These same tiles will be used for all clustering algorithms and cluster numbers\n"
# )

In [16]:
# # Create data for all models at once (same tiles, same activations)
# print("Processing all tiles with all clustering models...")
# all_datasets = create_data(clustering_models, tile_samples)

# print(f"\nCreated {len(all_datasets)} separate datasets:")
# for model_name, data in all_datasets.items():
#     print(f"  {model_name}: {data.shape}")

In [17]:
# # Save each dataset to a separate CSV file
# print("\nSaving datasets...")
# for model_name, data in all_datasets.items():
#     filename = f"data_{model_name.lower()}.csv"
#     data.to_csv(filename, index=False)
#     print(f"  Saved {filename}")

In [18]:
# Load all separate datasets
all_datasets = {}

print("Loading separate datasets:")
for clustering_algorithm in CLUSTERING_ALGORITHMS:
    for num_clusters in CLUSTER_NUMBERS:
        model_name = f"{clustering_algorithm}_{num_clusters}"
        filename = f"data_{model_name.lower()}.csv"
        data = pd.read_csv(filename)
        all_datasets[model_name] = data
        print(f"  {model_name}: {data.shape}")

print(f"\nTotal datasets loaded: {len(all_datasets)}")

# Create combinations list from the datasets
combinations = []
for _, data in all_datasets.items():
    alg = data["clustering_algorithm"].iloc[0]
    n_clust = data["num_clusters"].iloc[0]
    combinations.append({"clustering_algorithm": alg, "num_clusters": n_clust})

combinations = pd.DataFrame(combinations).drop_duplicates()
print("\nAvailable combinations:")
print(combinations.to_string(index=False))

Loading separate datasets:
  NMF_4: (10000, 137)
  NMF_6: (10000, 195)
  NMF_8: (10000, 253)
  KMeans_4: (10000, 137)
  KMeans_6: (10000, 195)
  KMeans_8: (10000, 253)

Total datasets loaded: 6

Available combinations:
clustering_algorithm  num_clusters
                 NMF             4
                 NMF             6
                 NMF             8
              KMeans             4
              KMeans             6
              KMeans             8


In [ ]:
# Constants
CAM_METHODS = ["grad_cam", "grad_cam_pp", "layer_cam"]
MASK_TYPES = ["carcinoma", "another_pathology"]
MASK_DISPLAYS = {"carcinoma": "Carcinoma", "another_pathology": "Another Pathology"}


# Helper functions for visualization
def get_method_display(method: str) -> str:
    """Get display name for CAM method."""
    return {
        "grad_cam": "Grad-CAM",
        "grad_cam_pp": "Grad-CAM++",
        "layer_cam": "Layer-CAM",
    }.get(method, method)


def calculate_iou(
    data: pd.DataFrame, intersection_col: str, union_col: str, iou_col: str
):
    """Calculate IoU from intersection and union columns."""
    mask = data[union_col].notna() & (data[union_col] > 0)
    data.loc[mask, iou_col] = (
        data.loc[mask, intersection_col] / data.loc[mask, union_col]
    )


def create_boxplot_by_label(
    data: pd.DataFrame,
    columns: list[str],
    labels: list[str],
    title: str,
    ylabel: str,
    filename: str,
    figsize: tuple = (14, 6),
):
    """Create boxplot comparing two labels."""
    cluster_data_label0 = []
    cluster_data_label1 = []

    for col in columns:
        data_0 = data[data["predicted_label"] == 0][col].dropna()
        data_1 = data[data["predicted_label"] == 1][col].dropna()
        cluster_data_label0.append(data_0)
        cluster_data_label1.append(data_1)

    positions = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=figsize)
    bp0 = plt.boxplot(
        cluster_data_label0,
        positions=positions - width / 2,
        widths=width,
        tick_labels=labels,
        patch_artist=True,
    )
    bp1 = plt.boxplot(
        cluster_data_label1,
        positions=positions + width / 2,
        widths=width,
        patch_artist=True,
    )

    for patch in bp0["boxes"]:
        patch.set_facecolor("lightblue")
    for patch in bp1["boxes"]:
        patch.set_facecolor("lightcoral")

    plt.title(title)
    plt.xlabel("Cluster" if labels and "Cluster" in labels[0] else "Category")
    plt.ylabel(ylabel)
    plt.xticks(positions, labels, rotation=45)
    plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")


def create_boxplot_all_data(
    data: pd.DataFrame,
    columns: list[str],
    labels: list[str],
    title: str,
    ylabel: str,
    filename: str,
    figsize: tuple = (12, 6),
):
    """Create boxplot for all data."""
    cluster_data = []
    for col in columns:
        col_data = data[col].dropna()
        cluster_data.append(col_data)

    plt.figure(figsize=figsize)
    plt.boxplot(cluster_data, tick_labels=labels)
    plt.title(title)
    plt.xlabel("Cluster" if labels and "Cluster" in labels[0] else "Category")
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")


def create_metric_plots(
    data: pd.DataFrame,
    method: str,
    metric_suffix: str,
    metric_display: str,
    num_clusters: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
):
    """Create both by-label and all-data plots for a metric."""
    method_display = get_method_display(method)
    cluster_labels = [f"Cluster {i + 1}" for i in range(num_clusters)]
    columns = [f"{metric_suffix}_{method}_{i}" for i in range(num_clusters)]

    # By label plot
    create_boxplot_by_label(
        data,
        columns,
        cluster_labels,
        f"{method_display} {metric_display} by Cluster - {alg} ({n_clust} clusters)",
        metric_display,
        f"{figures_dir}/{metric_suffix}_by_label_{method}.png",
    )

    # All data plot
    create_boxplot_all_data(
        data,
        columns,
        cluster_labels,
        f"{method_display} {metric_display} by Cluster (All Data) - {alg} ({n_clust} clusters)",
        metric_display,
        f"{figures_dir}/{metric_suffix}_all_data_{method}.png",
    )


def create_annotation_mask_plots(
    data: pd.DataFrame,
    mask_type: str,
    mask_display: str,
    num_clusters: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
):
    """Create hard and soft IoU plots for annotation masks."""
    data_with_mask = data[data[f"has_{mask_type}_mask"] == 1]

    if len(data_with_mask) == 0:
        return

    # Hard IoU: Combine all CAM methods + all clusters
    hard_data = []
    hard_labels = []

    # Add CAM methods
    for method in CAM_METHODS:
        method_display = get_method_display(method)
        iou_col = f"iou_{method}_{mask_type}"
        iou_data = data_with_mask[iou_col].dropna()
        if len(iou_data) > 0:
            hard_data.append(iou_data)
            hard_labels.append(method_display)

    # Add clusters
    for i in range(num_clusters):
        iou_col = f"iou_cluster_{i}_{mask_type}"
        iou_data = data_with_mask[iou_col].dropna()
        if len(iou_data) > 0:
            hard_data.append(iou_data)
            hard_labels.append(f"Cluster {i + 1}")

    if len(hard_data) > 0:
        plt.figure(figsize=(14, 6))
        plt.boxplot(hard_data, tick_labels=hard_labels)
        plt.title(f"IoU (Hard) with {mask_display} Mask - {alg} ({n_clust} clusters)")
        plt.xlabel("Method/Cluster")
        plt.ylabel("IoU")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(
            f"{figures_dir}/iou_hard_{mask_type}.png", dpi=300, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved figure: {figures_dir}/iou_hard_{mask_type}.png")

    # Soft IoU: Only clusters
    soft_data = []
    soft_labels = []
    for i in range(num_clusters):
        iou_col = f"iou_soft_cluster_{i}_{mask_type}"
        iou_data = data_with_mask[iou_col].dropna()
        if len(iou_data) > 0:
            soft_data.append(iou_data)
            soft_labels.append(f"Cluster {i + 1}")

    if len(soft_data) > 0:
        plt.figure(figsize=(12, 6))
        plt.boxplot(soft_data, tick_labels=soft_labels)
        plt.title(f"IoU (Soft) with {mask_display} Mask - {alg} ({n_clust} clusters)")
        plt.xlabel("Cluster")
        plt.ylabel("IoU")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(
            f"{figures_dir}/iou_soft_{mask_type}.png", dpi=300, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved figure: {figures_dir}/iou_soft_{mask_type}.png")


def create_cam_cluster_table(
    data: pd.DataFrame,
    method: str,
    label_filter: int | None,
    num_clusters: int,
    total_pixels: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
) -> pd.DataFrame:
    """Create statistics table for CAM method and clusters."""
    method_display = get_method_display(method)

    if label_filter is None:
        label_name = "Overall"
        label_suffix = "all"
        label_data = data
    else:
        label_name = f"Label {label_filter}"
        label_suffix = f"label_{label_filter}"
        label_data = data[data["predicted_label"] == label_filter]

    print(f"\n{method_display} - {label_name}:")
    print("-" * 60)

    table_data = []
    for i in range(num_clusters):
        row = {"Cluster": i + 1}

        # Weight Sum
        weight_col = f"weight_sum_{i}"
        weight_values = label_data[weight_col].dropna()
        row["Weight Sum"] = (
            f"{weight_values.mean():.4f} ± {weight_values.std():.4f}"
            if len(weight_values) > 0
            else "N/A"
        )

        # Cluster Count
        count_col = f"cluster_count_{i}"
        count_values = label_data[count_col].dropna()
        row["Cluster Count"] = (
            f"{count_values.mean():.4f} ± {count_values.std():.4f}"
            if len(count_values) > 0
            else "N/A"
        )

        # Normalized Cluster Count
        if len(count_values) > 0:
            normalized_counts = count_values / total_pixels
            row["Norm Count"] = (
                f"{normalized_counts.mean():.4f} ± {normalized_counts.std():.4f}"
            )
        else:
            row["Norm Count"] = "N/A"

        # Intersection / Total Pixels (Hard)
        intersection_col = f"intersection_count_{method}_{i}"
        intersection_values = label_data[intersection_col].dropna()
        if len(intersection_values) > 0:
            normalized_intersection = intersection_values / total_pixels
            row["Intersection/Pixels"] = (
                f"{normalized_intersection.mean():.4f} ± {normalized_intersection.std():.4f}"
            )
        else:
            row["Intersection/Pixels"] = "N/A"

        # Intersection / Total Pixels (Soft)
        intersection_soft_col = f"intersection_soft_count_{method}_{i}"
        intersection_soft_values = label_data[intersection_soft_col].dropna()
        if len(intersection_soft_values) > 0:
            normalized_intersection_soft = intersection_soft_values / total_pixels
            row["Intersection/Pixels (Soft)"] = (
                f"{normalized_intersection_soft.mean():.4f} ± {normalized_intersection_soft.std():.4f}"
            )
        else:
            row["Intersection/Pixels (Soft)"] = "N/A"

        # IoU
        iou_col = f"iou_{method}_{i}"
        iou_values = label_data[iou_col].dropna()
        row["IoU"] = (
            f"{iou_values.mean():.4f} ± {iou_values.std():.4f}"
            if len(iou_values) > 0
            else "N/A"
        )

        # Soft IoU
        soft_iou_col = f"iou_soft_{method}_{i}"
        soft_iou_values = label_data[soft_iou_col].dropna()
        row["Soft IoU"] = (
            f"{soft_iou_values.mean():.4f} ± {soft_iou_values.std():.4f}"
            if len(soft_iou_values) > 0
            else "N/A"
        )

        # Correlation
        corr_col = f"correlation_{method}_{i}"
        corr_values = label_data[corr_col].dropna()
        row["Correlation"] = (
            f"{corr_values.mean():.4f} ± {corr_values.std():.4f}"
            if len(corr_values) > 0
            else "N/A"
        )

        # L2 Distance
        l2_col = f"l2_distance_{method}_{i}"
        l2_values = label_data[l2_col].dropna()
        row["L2 Distance"] = (
            f"{l2_values.mean():.4f} ± {l2_values.std():.4f}"
            if len(l2_values) > 0
            else "N/A"
        )

        table_data.append(row)

    if table_data:
        table_df = pd.DataFrame(table_data)
        display(table_df)
        filename = f"{figures_dir}/table_{method}_{label_suffix}.csv"
        table_df.to_csv(filename, index=False)
        print(f"Saved table: {filename}")
        return table_df
    return pd.DataFrame()


def create_cam_iou_correlation_table(
    data: pd.DataFrame,
    method: str,
    num_clusters: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
) -> pd.DataFrame:
    """Create table showing IoU (all data) and Correlation (Label 0, Label 1) for each CAM method."""
    _ = get_method_display(method)

    def get_statistic_string(values: pd.Series) -> str:
        """Get formatted statistic string (mean ± std) or N/A."""
        if len(values) > 0:
            return f"{values.mean():.4f} ± {values.std():.4f}"
        return "N/A"

    table_rows = []

    for i in range(num_clusters):
        row = {"Cluster": i + 1}

        # IoU (All Data)
        iou_col = f"iou_{method}_{i}"
        iou_values_all = data[iou_col].dropna()
        row["IoU (All Data)"] = get_statistic_string(iou_values_all)

        # Correlation (Label 0)
        data_label0 = data[data["predicted_label"] == 0]
        corr_col = f"correlation_{method}_{i}"
        corr_values_label0 = data_label0[corr_col].dropna()
        row["Correlation (Label 0)"] = get_statistic_string(corr_values_label0)

        # Correlation (Label 1)
        data_label1 = data[data["predicted_label"] == 1]
        corr_values_label1 = data_label1[corr_col].dropna()
        row["Correlation (Label 1)"] = get_statistic_string(corr_values_label1)

        table_rows.append(row)

    if table_rows:
        table_df = pd.DataFrame(table_rows)
        display(table_df)
        filename = f"{figures_dir}/table_{method}_iou_correlation.csv"
        table_df.to_csv(filename, index=False)
        print(f"Saved table: {filename}")
        return table_df
    return pd.DataFrame()


def create_cam_cluster_comparison_table(
    data: pd.DataFrame,
    method: str,
    num_clusters: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
) -> pd.DataFrame:
    """Create table comparing CAM-cluster metrics across labels (Label 0, Label 1)."""
    _ = get_method_display(method)

    def get_statistic_string(values: pd.Series) -> str:
        """Get formatted statistic string (mean ± std) or N/A."""
        if len(values) > 0:
            return f"{values.mean():.4f} ± {values.std():.4f}"
        return "N/A"

    table_rows = []

    for i in range(num_clusters):
        row = {"Cluster": i + 1}

        # Label 0 metrics
        data_label0 = data[data["predicted_label"] == 0]

        # IoU
        iou_col = f"iou_{method}_{i}"
        iou_values_label0 = data_label0[iou_col].dropna()
        row["Label 0 IoU"] = get_statistic_string(iou_values_label0)

        # Correlation
        corr_col = f"correlation_{method}_{i}"
        corr_values_label0 = data_label0[corr_col].dropna()
        row["Label 0 Correlation"] = get_statistic_string(corr_values_label0)

        # L2 Distance
        l2_col = f"l2_distance_{method}_{i}"
        l2_values_label0 = data_label0[l2_col].dropna()
        row["Label 0 L2 Distance"] = get_statistic_string(l2_values_label0)

        # Label 1 metrics
        data_label1 = data[data["predicted_label"] == 1]

        # IoU
        iou_values_label1 = data_label1[iou_col].dropna()
        row["Label 1 IoU"] = get_statistic_string(iou_values_label1)

        # Correlation
        corr_values_label1 = data_label1[corr_col].dropna()
        row["Label 1 Correlation"] = get_statistic_string(corr_values_label1)

        # L2 Distance
        l2_values_label1 = data_label1[l2_col].dropna()
        row["Label 1 L2 Distance"] = get_statistic_string(l2_values_label1)

        table_rows.append(row)

    if table_rows:
        table_df = pd.DataFrame(table_rows)
        display(table_df)
        filename = f"{figures_dir}/table_{method}_cluster_comparison.csv"
        table_df.to_csv(filename, index=False)
        print(f"Saved table: {filename}")
        return table_df
    return pd.DataFrame()


def create_norm_count_comparison_table(
    data: pd.DataFrame,
    num_clusters: int,
    total_pixels: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
) -> pd.DataFrame:
    """Create table comparing normalized cluster counts across labels (All Data, Label 0, Label 1)."""

    def get_statistic_string(values: pd.Series) -> str:
        """Get formatted statistic string (mean ± std) or N/A."""
        if len(values) > 0:
            return f"{values.mean():.4f} ± {values.std():.4f}"
        return "N/A"

    table_rows = []

    for i in range(num_clusters):
        row = {"Cluster": i + 1}

        count_col = f"cluster_count_{i}"

        # All Data
        count_values_all = data[count_col].dropna()
        if len(count_values_all) > 0:
            normalized_counts_all = count_values_all / total_pixels
            row["All Data"] = get_statistic_string(normalized_counts_all)
        else:
            row["All Data"] = "N/A"

        # Label 0
        data_label0 = data[data["predicted_label"] == 0]
        count_values_label0 = data_label0[count_col].dropna()
        if len(count_values_label0) > 0:
            normalized_counts_label0 = count_values_label0 / total_pixels
            row["Label 0"] = get_statistic_string(normalized_counts_label0)
        else:
            row["Label 0"] = "N/A"

        # Label 1
        data_label1 = data[data["predicted_label"] == 1]
        count_values_label1 = data_label1[count_col].dropna()
        if len(count_values_label1) > 0:
            normalized_counts_label1 = count_values_label1 / total_pixels
            row["Label 1"] = get_statistic_string(normalized_counts_label1)
        else:
            row["Label 1"] = "N/A"

        table_rows.append(row)

    if table_rows:
        table_df = pd.DataFrame(table_rows)
        display(table_df)
        filename = f"{figures_dir}/table_norm_count_comparison.csv"
        table_df.to_csv(filename, index=False)
        print(f"Saved table: {filename}")
        return table_df
    return pd.DataFrame()


def create_carcinoma_iou_comparison_table(
    data: pd.DataFrame,
    num_clusters: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
) -> pd.DataFrame:
    """Create table comparing Carcinoma IoU across labels (Label 0, Label 1, All Data)."""
    data_with_mask = data[data["has_carcinoma_mask"] == 1]

    if len(data_with_mask) == 0:
        return pd.DataFrame()

    mask_type = "carcinoma"

    def get_statistic_string(values: pd.Series) -> str:
        """Get formatted statistic string (mean ± std) or N/A."""
        if len(values) > 0:
            return f"{values.mean():.4f} ± {values.std():.4f}"
        return "N/A"

    table_rows = []

    # Add CAM method rows
    for method in CAM_METHODS:
        method_display = get_method_display(method)
        row = {"Method/Cluster": method_display}

        iou_col = f"iou_{method}_{mask_type}"

        # All Data
        iou_all = data_with_mask[iou_col].dropna()
        row["All Data"] = get_statistic_string(iou_all)

        # Label 0
        iou_label0 = data_with_mask[data_with_mask["predicted_label"] == 0][
            iou_col
        ].dropna()
        row["Label 0"] = get_statistic_string(iou_label0)

        # Label 1
        iou_label1 = data_with_mask[data_with_mask["predicted_label"] == 1][
            iou_col
        ].dropna()
        row["Label 1"] = get_statistic_string(iou_label1)

        table_rows.append(row)

    # Add cluster rows
    for i in range(num_clusters):
        row = {"Method/Cluster": f"Cluster {i + 1}"}

        iou_col = f"iou_cluster_{i}_{mask_type}"

        # All Data
        iou_all = data_with_mask[iou_col].dropna()
        row["All Data"] = get_statistic_string(iou_all)

        # Label 0
        iou_label0 = data_with_mask[data_with_mask["predicted_label"] == 0][
            iou_col
        ].dropna()
        row["Label 0"] = get_statistic_string(iou_label0)

        # Label 1
        iou_label1 = data_with_mask[data_with_mask["predicted_label"] == 1][
            iou_col
        ].dropna()
        row["Label 1"] = get_statistic_string(iou_label1)

        table_rows.append(row)

    if table_rows:
        table_df = pd.DataFrame(table_rows)
        display(table_df)
        filename = f"{figures_dir}/table_carcinoma_iou_comparison.csv"
        table_df.to_csv(filename, index=False)
        print(f"Saved table: {filename}")
        return table_df
    return pd.DataFrame()


def create_annotation_mask_table(
    data: pd.DataFrame,
    mask_type: str,
    mask_display: str,
    num_clusters: int,
    figures_dir: str,
    alg: str,
    n_clust: int,
    label_filter: int | None = None,
) -> pd.DataFrame:
    """Create comprehensive table for annotation mask metrics."""
    data_with_mask = data[data[f"has_{mask_type}_mask"] == 1]

    if len(data_with_mask) == 0:
        return pd.DataFrame()

    # Filter by predicted label if specified
    if label_filter is not None:
        data_with_mask = data_with_mask[
            data_with_mask["predicted_label"] == label_filter
        ]
        if len(data_with_mask) == 0:
            return pd.DataFrame()
        label_name = f"Predicted Label {label_filter}"
        label_suffix = f"label_{label_filter}"
    else:
        label_name = "All Data"
        label_suffix = "all"

    total_pixels = 32 * 32  # Total number of pixels

    print(f"\n{mask_display} Mask Metrics - {label_name} (n={len(data_with_mask)}):")
    print("-" * 60)

    table_rows = []

    # Add CAM method rows
    for method in CAM_METHODS:
        method_display = get_method_display(method)
        row = {"Method/Cluster": method_display}

        # Intersection / Total Pixels
        intersection_col = f"intersection_count_{method}_{mask_type}"
        intersection_values = data_with_mask[intersection_col].dropna()
        if len(intersection_values) > 0:
            normalized_intersection = intersection_values / total_pixels
            row["Intersection/Pixels"] = (
                f"{normalized_intersection.mean():.4f} ± {normalized_intersection.std():.4f}"
            )
        else:
            row["Intersection/Pixels"] = "N/A"

        iou_col = f"iou_{method}_{mask_type}"
        iou_values = data_with_mask[iou_col].dropna()
        row["IoU (Mean ± Std)"] = (
            f"{iou_values.mean():.4f} ± {iou_values.std():.4f}"
            if len(iou_values) > 0
            else "N/A"
        )
        row["IoU Soft (Mean ± Std)"] = "N/A"
        table_rows.append(row)

    # Add cluster rows
    for i in range(num_clusters):
        row = {"Method/Cluster": f"Cluster {i + 1}"}

        # Intersection / Total Pixels (Hard)
        intersection_col = f"intersection_count_cluster_{i}_{mask_type}"
        intersection_values = data_with_mask[intersection_col].dropna()
        if len(intersection_values) > 0:
            normalized_intersection = intersection_values / total_pixels
            row["Intersection/Pixels"] = (
                f"{normalized_intersection.mean():.4f} ± {normalized_intersection.std():.4f}"
            )
        else:
            row["Intersection/Pixels"] = "N/A"

        # Intersection / Total Pixels (Soft)
        intersection_soft_col = f"intersection_soft_count_cluster_{i}_{mask_type}"
        intersection_soft_values = data_with_mask[intersection_soft_col].dropna()
        if len(intersection_soft_values) > 0:
            normalized_intersection_soft = intersection_soft_values / total_pixels
            row["Intersection/Pixels (Soft)"] = (
                f"{normalized_intersection_soft.mean():.4f} ± {normalized_intersection_soft.std():.4f}"
            )
        else:
            row["Intersection/Pixels (Soft)"] = "N/A"

        iou_col = f"iou_cluster_{i}_{mask_type}"
        iou_values = data_with_mask[iou_col].dropna()
        row["IoU (Mean ± Std)"] = (
            f"{iou_values.mean():.4f} ± {iou_values.std():.4f}"
            if len(iou_values) > 0
            else "N/A"
        )

        soft_iou_col = f"iou_soft_cluster_{i}_{mask_type}"
        soft_iou_values = data_with_mask[soft_iou_col].dropna()
        row["IoU Soft (Mean ± Std)"] = (
            f"{soft_iou_values.mean():.4f} ± {soft_iou_values.std():.4f}"
            if len(soft_iou_values) > 0
            else "N/A"
        )
        table_rows.append(row)

    if table_rows:
        table_df = pd.DataFrame(table_rows)
        display(table_df)
        filename = f"{figures_dir}/table_{mask_type}_{label_suffix}.csv"
        table_df.to_csv(filename, index=False)
        print(f"Saved table: {filename}")
        return table_df
    return pd.DataFrame()


base_figures_dir = "figures"
os.makedirs(base_figures_dir, exist_ok=True)

# Process each clustering model combination separately
for _, row in combinations.iterrows():
    alg = row["clustering_algorithm"]
    n_clust = row["num_clusters"]
    model_name = f"{alg}_{n_clust}"

    # Create subdirectory for this combination
    figures_dir = f"{base_figures_dir}/{model_name}"
    os.makedirs(figures_dir, exist_ok=True)

    print(f"\n{'=' * 80}")
    print(f"Processing: {alg} with {n_clust} clusters")
    print(f"{'=' * 80}\n")

    # Get the dataset for this combination
    data = all_datasets[model_name].copy()

    num_clusters = n_clust

    # y is the sigmoid of output
    data["y"] = 1 / (1 + np.exp(-data["output"]))

    # print count of labels - either 0 or 1
    print("Label distribution:")
    display(data["label"].value_counts())

    plt.hist(data["y"])
    plt.title(
        f"Distribution of predicted probabilities (y) - {alg} ({n_clust} clusters)"
    )
    plt.xlabel("y")
    plt.ylabel("Frequency")
    filename = f"{figures_dir}/hist_predicted_prob.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")

    # Compute prediction metrics
    data["predicted_label"] = (data["y"] > 0.94).astype(int)

    print(f"\nPrediction Metrics - {alg} with {n_clust} clusters:")
    print("=" * 60)
    print(f"Number of samples: {len(data)}")
    display(pd.crosstab(data["predicted_label"], data["label"]))
    print(f"Accuracy: {accuracy_score(data['label'], data['predicted_label'])}")
    print(f"Precision: {precision_score(data['label'], data['predicted_label'])}")
    print(f"Recall: {recall_score(data['label'], data['predicted_label'])}")
    print(f"F1 Score: {f1_score(data['label'], data['predicted_label'])}")
    print(f"AUC Score: {roc_auc_score(data['label'], data['y'])}")

    # Add IoU columns for each method and cluster
    for method in CAM_METHODS:
        for i in range(num_clusters):
            calculate_iou(
                data,
                f"intersection_count_{method}_{i}",
                f"union_count_{method}_{i}",
                f"iou_{method}_{i}",
            )

    # Fit linear regression model
    print(f"\nFitting linear regression model - {alg} ({n_clust} clusters):")
    print("=" * 60)

    X = data[[f"weight_sum_{i}" for i in range(num_clusters)]]
    y = data["output"]

    model = LinearRegression()
    model.fit(X, y)

    print(f"Coefficients: {model.coef_}")
    print(f"Intercept: {model.intercept_}")
    print(f"R-squared: {model.score(X, y)}")

    # Fit linear regression model using cluster counts
    print(
        f"\nFitting linear regression model (using cluster counts) - {alg} ({n_clust} clusters):"
    )
    print("=" * 60)

    X_clusters = data[[f"cluster_count_{i}" for i in range(num_clusters)]]
    y = data["output"]

    model_clusters = LinearRegression()
    model_clusters.fit(X_clusters, y)

    print(f"Coefficients: {model_clusters.coef_}")
    print(f"Intercept: {model_clusters.intercept_}")
    print(f"R-squared: {model_clusters.score(X_clusters, y)}")

    # Visualize weight distributions by cluster
    cluster_labels = [f"Cluster {i + 1}" for i in range(num_clusters)]
    weight_cols = [f"weight_sum_{i}" for i in range(num_clusters)]

    create_boxplot_by_label(
        data,
        weight_cols,
        cluster_labels,
        f"Weight Sum by Cluster - {alg} ({n_clust} clusters)",
        "Weight Sum",
        f"{figures_dir}/weight_sum_by_label.png",
    )

    create_boxplot_all_data(
        data,
        weight_cols,
        cluster_labels,
        f"Weight Sum by Cluster (All Data) - {alg} ({n_clust} clusters)",
        "Weight Sum",
        f"{figures_dir}/weight_sum_all_data.png",
    )

    # Visualize cluster count distributions by cluster
    count_cols = [f"cluster_count_{i}" for i in range(num_clusters)]

    create_boxplot_by_label(
        data,
        count_cols,
        cluster_labels,
        f"Cluster Count by Cluster - {alg} ({n_clust} clusters)",
        "Cluster Count",
        f"{figures_dir}/cluster_count_by_label.png",
    )

    create_boxplot_all_data(
        data,
        count_cols,
        cluster_labels,
        f"Cluster Count by Cluster (All Data) - {alg} ({n_clust} clusters)",
        "Cluster Count",
        f"{figures_dir}/cluster_count_all_data.png",
    )

    # Add soft IoU columns for each method and cluster
    for method in CAM_METHODS:
        for i in range(num_clusters):
            calculate_iou(
                data,
                f"intersection_soft_count_{method}_{i}",
                f"union_soft_count_{method}_{i}",
                f"iou_soft_{method}_{i}",
            )

    # Add IoU columns for CAMs with annotation masks
    for method in CAM_METHODS:
        for mask_type in MASK_TYPES:
            calculate_iou(
                data,
                f"intersection_count_{method}_{mask_type}",
                f"union_count_{method}_{mask_type}",
                f"iou_{method}_{mask_type}",
            )

    # Add IoU columns for clusters with annotation masks
    for i in range(num_clusters):
        for mask_type in MASK_TYPES:
            # Hard cluster mask
            calculate_iou(
                data,
                f"intersection_count_cluster_{i}_{mask_type}",
                f"union_count_cluster_{i}_{mask_type}",
                f"iou_cluster_{i}_{mask_type}",
            )
            # Soft cluster mask
            calculate_iou(
                data,
                f"intersection_soft_count_cluster_{i}_{mask_type}",
                f"union_soft_count_cluster_{i}_{mask_type}",
                f"iou_soft_cluster_{i}_{mask_type}",
            )

    # Visualize for all CAM methods
    metrics = [
        ("iou", "IoU"),
        ("iou_soft", "Soft IoU"),
        ("correlation", "Correlation"),
        ("l2_distance", "L2 Distance"),
    ]

    for method in CAM_METHODS:
        for metric_suffix, metric_display in metrics:
            create_metric_plots(
                data,
                method,
                metric_suffix,
                metric_display,
                num_clusters,
                figures_dir,
                alg,
                n_clust,
            )

    # Visualize IoU with annotation masks - 2 plots per mask type (hard and soft)
    for mask_type in MASK_TYPES:
        mask_display = MASK_DISPLAYS[mask_type]
        create_annotation_mask_plots(
            data, mask_type, mask_display, num_clusters, figures_dir, alg, n_clust
        )

    # Create tables split by CAM method and label
    print(f"\n{'=' * 60}")
    print(f"Statistics (Mean ± Std) - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")

    total_pixels = 32 * 32  # Total number of pixels

    # For each CAM method
    for method in CAM_METHODS:
        # For each label split (Overall, Label 0, Label 1)
        for label_filter in [None, 0, 1]:
            create_cam_cluster_table(
                data,
                method,
                label_filter,
                num_clusters,
                total_pixels,
                figures_dir,
                alg,
                n_clust,
            )

    # Create CAM-cluster comparison tables (Label 0 vs Label 1)
    print(f"\n{'=' * 60}")
    print(f"CAM-Cluster Comparison (Label 0 vs Label 1) - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")
    for method in CAM_METHODS:
        create_cam_cluster_comparison_table(
            data,
            method,
            num_clusters,
            figures_dir,
            alg,
            n_clust,
        )

    # Create CAM IoU and Correlation tables
    print(f"\n{'=' * 60}")
    print(
        f"CAM IoU (All Data) and Correlation (Label 0, Label 1) - {alg} ({n_clust} clusters)"
    )
    print(f"{'=' * 60}\n")
    for method in CAM_METHODS:
        create_cam_iou_correlation_table(
            data,
            method,
            num_clusters,
            figures_dir,
            alg,
            n_clust,
        )

    # Create normalized cluster count comparison table
    print(f"\n{'=' * 60}")
    print(f"Normalized Cluster Count Comparison - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")
    create_norm_count_comparison_table(
        data,
        num_clusters,
        total_pixels,
        figures_dir,
        alg,
        n_clust,
    )

    # Print statistics for annotation mask metrics
    print(f"\n{'=' * 60}")
    print(f"Annotation Mask Statistics - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")

    # Comprehensive tables for annotation masks
    # For each annotation mask type, create 3 tables: all data, predicted label 0, predicted label 1
    for mask_type in MASK_TYPES:
        mask_display = MASK_DISPLAYS[mask_type]
        for label_filter in [None, 0, 1]:
            create_annotation_mask_table(
                data,
                mask_type,
                mask_display,
                num_clusters,
                figures_dir,
                alg,
                n_clust,
                label_filter,
            )

    # Create Carcinoma IoU comparison table
    print(f"\n{'=' * 60}")
    print(f"Carcinoma IoU Comparison - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")
    create_carcinoma_iou_comparison_table(
        data,
        num_clusters,
        figures_dir,
        alg,
        n_clust,
    )

    print(f"\n{'=' * 80}\n")


Processing: NMF with 4 clusters

Label distribution:


label
0.0    8369
1.0    1631
Name: count, dtype: int64

Saved figure: figures/NMF_4/hist_predicted_prob.png

Prediction Metrics - NMF with 4 clusters:
Number of samples: 10000


label,0.0,1.0
predicted_label,,
0,8093,124
1,276,1507


Accuracy: 0.96
Precision: 0.8452047111609646
Recall: 0.923973022685469
F1 Score: 0.8828353837141183
AUC Score: 0.9844677288867657

Fitting linear regression model - NMF (4 clusters):
Coefficients: [-0.03855283  0.04259869 -0.02857835  0.42907042]
Intercept: -2.761856093920791
R-squared: 0.7580769296044544

Fitting linear regression model (using cluster counts) - NMF (4 clusters):
Coefficients: [-0.01626165  0.01086608 -0.01284096  0.01823652]
Intercept: 8.295282748419986
R-squared: 0.7395264688422954
Saved figure: figures/NMF_4/weight_sum_by_label.png
Saved figure: figures/NMF_4/weight_sum_all_data.png
Saved figure: figures/NMF_4/cluster_count_by_label.png
Saved figure: figures/NMF_4/cluster_count_all_data.png
Saved figure: figures/NMF_4/iou_by_label_grad_cam.png
Saved figure: figures/NMF_4/iou_all_data_grad_cam.png
Saved figure: figures/NMF_4/iou_soft_by_label_grad_cam.png
Saved figure: figures/NMF_4/iou_soft_all_data_grad_cam.png
Saved figure: figures/NMF_4/correlation_by_label_grad_

...

Saved table: figures/KMeans_8/table_carcinoma_label_0.csv

Carcinoma Mask Metrics - Predicted Label 1 (n=1596):
------------------------------------------------------------


,Method/Cluster,Intersection/Pixels,IoU (Mean ± Std),IoU Soft (Mean ± Std),Intersection/Pixels (Soft)
0,Grad-CAM,0.4723 ± 0.2525,0.5828 ± 0.2135,N/A,NaN
1,Grad-CAM++,0.5009 ± 0.2592,0.6111 ± 0.2168,N/A,NaN
2,Layer-CAM,0.0565 ± 0.0171,0.0923 ± 0.0449,N/A,NaN
3,Cluster 1,0.1600 ± 0.1122,0.1728 ± 0.0760,0.3873 ± 0.1240,0.6908 ± 0.2950
4,Cluster 2,0.4384 ± 0.2260,0.2924 ± 0.1122,0.3873 ± 0.1240,0.6908 ± 0.2950
5,Cluster 3,0.0009 ± 0.0045,0.0016 ± 0.0074,0.3873 ± 0.1240,0.6908 ± 0.2950
6,Cluster 4,0.0213 ± 0.0348,0.0276 ± 0.0406,0.3873 ± 0.1240,0.6908 ± 0.2950
7,Cluster 5,0.0600 ± 0.0854,0.0676 ± 0.0742,0.3873 ± 0.1240,0.6908 ± 0.2950
8,Cluster 6,0.0036 ± 0.0119,0.0055 ± 0.0175,0.3873 ± 0.1240,0.6908 ± 0.2950
9,Cluster 7,0.0020 ± 0.0061,0.0028 ± 0.0084,0.3873 ± 0.1240,0.6908 ± 0.2950


Saved table: figures/KMeans_8/table_carcinoma_label_1.csv

Carcinoma IoU Comparison - KMeans (8 clusters)



,Method/Cluster,All Data,Label 0,Label 1
0,Grad-CAM,0.5311 ± 0.2455,0.1946 ± 0.1593,0.5828 ± 0.2135
1,Grad-CAM++,0.5584 ± 0.2502,0.2150 ± 0.1688,0.6111 ± 0.2168
2,Layer-CAM,0.0920 ± 0.0526,0.0903 ± 0.0879,0.0923 ± 0.0449
3,Cluster 1,0.1557 ± 0.0866,0.0442 ± 0.0659,0.1728 ± 0.0760
4,Cluster 2,0.2738 ± 0.1261,0.1526 ± 0.1434,0.2924 ± 0.1122
5,Cluster 3,0.0017 ± 0.0079,0.0027 ± 0.0103,0.0016 ± 0.0074
6,Cluster 4,0.0305 ± 0.0443,0.0493 ± 0.0600,0.0276 ± 0.0406
7,Cluster 5,0.0590 ± 0.0726,0.0035 ± 0.0131,0.0676 ± 0.0742
8,Cluster 6,0.0068 ± 0.0215,0.0152 ± 0.0374,0.0055 ± 0.0175
9,Cluster 7,0.0033 ± 0.0110,0.0070 ± 0.0211,0.0028 ± 0.0084


Saved table: figures/KMeans_8/table_carcinoma_iou_comparison.csv


